**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/catalystneuro/encore", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/kussovska-2025"], check=True)
    os.chdir("/content/encore/entries/kussovska-2025")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Kussovska et al. (2025) on dandiset 000469: the dPCA state-space claims

Paper. Kussovska D, Kim R, Rungratsameetaweemana N (2025). Neural heterogeneity shapes the
temporal structure of human working memory. bioRxiv,
[doi:10.1101/2025.10.31.684900](https://doi.org/10.1101/2025.10.31.684900). The claims file
(`claims/kussovska-2025.yaml`) was written from the bioRxiv full text posted on 1 November 2025.

Dandiset. [DANDI:000469](https://dandiarchive.org/dandiset/000469/0.231213.2047), version
0.231213.2047, the version the paper cites (see the header of `reproduce_kussovska_2025.ipynb`,
which tests the other claims and shows that the four published versions hold the same assets).

This notebook tests the three claims on dPCA (Fig. 3), apart from the main notebook so that each
stays within the time limit. It streams the same files, finds the same concept cells (the code's
bootstrap, seed 0, as in the main notebook) and runs the 100 resampling splits of the paper's code
in parallel, each dPCA fit on one thread.

| Claim | Tested here |
|---|---|
| fig3d-encoding-vs-maintenance-separability | tested |
| fig3abc-trajectory-complexity | tested, on the held-out trajectories of the fig3d fits |
| fig3h-early-vs-late-maintenance | tested |

Outside input: the code's load-matched trial table `data/new_trial_info.xlsx`
(github.com/dariakussovska/heterogeneitywm at commit 5144fe8), from which
08_dpca_matrices.py draws the 30 trials.

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
import scikit_posthocs as sp
from scipy import stats

import wm

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
T0 = time.time()

sessions = wm.load_all()
BY = {s["sub"]: s for s in sessions}
NT = wm.neuron_table(sessions)
T = {s["sub"]: wm.trial_frame(s) for s in sessions}
SPK = [wm.spikes_of(BY, r) for _, r in NT.iterrows()]
INFO = wm.code_table("data/new_trial_info.xlsx")
print(f"streamed {len(sessions)} Sternberg files of dandiset {wm.DANDISET_ID} version {wm.DANDISET_VERSION} "
      f"in {time.time() - T0:.0f} s; new_trial_info {INFO.shape}")

## Data check

What is loaded, and the link of the code's trial table to the dataset's trials (the main
notebook's data check covers the rest: units, regions, trial timing, clock and duplicates).

In [ ]:
tr = pd.concat(T.values())
print(f"{len(sessions)} sessions, {len(NT)} units; trials per load:", tr.load.value_counts().sort_index().to_dict())
print("units by region:", NT.region.value_counts().to_dict())
print(f"maintenance duration {tr.maint_dur.min():.3f} to {tr.maint_dur.max():.3f} s; second shortest "
      f"{np.sort(tr.maint_dur.values)[1]:.3f} s; first-encoding duration "
      f"{(tr.timestamps_Encoding1_end - tr.enc1).min():.3f} to {(tr.timestamps_Encoding1_end - tr.enc1).max():.3f} s")
ok, dmax = [], 0.0
for sub, t in T.items():
    x = INFO[INFO.subject_id == sub]
    m = t.set_index("trial").loc[x.trial_id.values]
    ok.append(bool((x.num_images_presented.values == m.load.values).all()))
    dmax = max(dmax, float(np.abs(x.RT.values - m.rt.values).max()))
print(f"new_trial_info, trial_id taken as the row of each session's trials table: loads agree in every session {all(ok)}; "
      f"largest RT difference {dmax:.2e} s")

## Concept cells

The set used downstream in the main notebook: the code's bootstrap (reading b of
concept-cells-count), seed 0.

In [ ]:
rng = np.random.default_rng(0)
res = []
for n in range(len(NT)):
    t = T[NT["sub"].iat[n]]
    r = wm.count_in(SPK[n], (t.enc1 + 0.2).values, (t.enc1 + 1.0).values) / 0.8
    res.append(wm.concept_test(r, t.img1.values, rng, reading="code"))
CONCEPT = np.array([s for _, _, s in res])
CELLS_C = np.where(CONCEPT)[0]
print(f"{len(CELLS_C)} concept cells in {len(set(NT['sub'].values[CELLS_C]))} participants; by region:",
      NT.region[CONCEPT].value_counts().to_dict())

## The dPCA design

From the criterion of fig3d and the code. 08_dpca_matrices.py draws 6 load-1 trials per image
from subject 14's rows of `new_trial_info.xlsx` (`sample(random_state=42)`), and for each neuron
takes its session's trials with those `new_trial_id`s; rates in 2 ms bins, z-scored with the
fixation rate, smoothed with a Gaussian of 60 bins, encoding 0 to 1 s from the first image and
maintenance 0 to 2.5 s (499 and 1249 bins, as the code's `np.arange` edges give). Trial readings:
(a) the same 30 trials in both epochs; (b) the code as written, whose maintenance trials are those
whose `trial_id` (not `new_trial_id`) is among the drawn ids, so that in sessions other than
subject 14 they are other trials, of any load. The splits are 100 stratified draws of 3 training
and 3 test trials per image; each split fits dPCA (Kobak's package, `labels="st"`, regularizer
chosen by its cross-validation, `protect=[]`) to the training trials of encoding and of
maintenance, and, for fig3h, to the early (bins 0 to 623) and late (624 to 1248) halves of
maintenance, each half with `protect=['t']` and with `protect=[]`. The measure is the mean over
the 10 image pairs of the distance between the time-averaged trajectories on the first five
stimulus dPCs (main_3d,supp_2a,b,i.py:56-66).

The code runs the splits in one sequence from `np.random.seed(1)`, which dPCA's own random draws
(its cross-validation folds and randomized SVD) also consume, so its splits cannot be repeated
outside its run. Here the 100 splits are drawn first from `default_rng(1)` and each split's fit is
seeded with `np.random.seed(1000 + split)`, a standard choice that lets them run in parallel.

In [ ]:
TRIALS = wm.dpca_trial_ids(INFO)
print("subject 14's drawn new_trial_ids by stimulus_index:", {int(s): ids.tolist() for s, ids in TRIALS})
tE, tDa = wm.dpca_tensors(CELLS_C, NT, T, SPK, INFO, "a")
_, tDb = wm.dpca_tensors(CELLS_C, NT, T, SPK, INFO, "b")
print("tensors (repetitions, neurons, stimuli, bins): encoding", tE.shape, "; maintenance", tDa.shape)
# how far reading (b) moves the maintenance trials
allids = np.concatenate([i for _, i in TRIALS])
mv = []
for sub in sorted(set(NT["sub"].values[CELLS_C])):
    t = T[sub].set_index("trial")
    a = INFO[(INFO.subject_id == sub) & INFO.new_trial_id.isin(allids)].trial_id.values
    b = allids[allids <= len(t)]
    mv.append({"sub": sub, "cells": int((NT["sub"].values[CELLS_C] == sub).sum()), "trials_a": len(a),
               "trials_b": len(b), "shared": len(set(a) & set(b)), "load1_in_b": int((t.loc[b].load == 1).sum())})
print(pd.DataFrame(mv).to_string(index=False))
print("maintenance traces that are all zero (no such trial or zero fixation s.d.): (a)",
      int((tDa == 0).all(-1).sum()), "(b)", int((tDb == 0).all(-1).sum()), "of", tDa.shape[0] * tDa.shape[1] * tDa.shape[2])

rng = np.random.default_rng(1)
SPLITS = [[rng.permutation(6) for _ in range(5)] for _ in range(100)]
print("distinct splits:", len({tuple(map(tuple, s)) for s in SPLITS}))
t0 = time.time()
RES = wm.pmap(wm.dpca_split, [(tE, {"a": tDa, "b": tDb}, s, 1000 + i) for i, s in enumerate(SPLITS)])
print(f"100 splits, {len(RES[0])} values each, in {(time.time() - t0) / 60:.1f} min on 4 processes")

## fig3d-encoding-vs-maintenance-separability

> Stimulus separation in dPCA space was higher during encoding than during maintenance, and higher
> than maintenance activity projected into the encoding space, while maintenance separation did
> not differ between its own space and the encoding space.

Kruskal-Wallis over the five groups of the code (the three test conditions and the two training
sets; the main reading) and over the three test conditions (a variant), then Dunn's test with BH
correction (`scikit_posthocs.posthoc_dunn(p_adjust="fdr_bh")`, as the code). Paper: H = 457.217
(tolerance 45.72), Dunn p 2.38e-38 (encoding against maintenance) and 1.25e-29 (against
maintenance projected into the encoding space).

In [ ]:
def kw_dunn(groups):
    names = list(groups)
    data = [np.asarray(groups[k]) for k in names]
    H, p = stats.kruskal(*data)
    d = sp.posthoc_dunn(data, p_adjust="fdr_bh")
    d.index = d.columns = names
    return H, p, d


def col(key):
    return np.array([r[key] for r in RES])


FIG3D = {}
for rd in ["a", "b"]:
    five = {"Enc_train": col("Enc_train"), "Maint_train": col((rd, "Maint_train")), "Enc_test": col("Enc_test"),
            "Maint_test": col((rd, "Maint_test")), "Maint onto Enc": col((rd, "Maint onto Enc"))}
    three = {k: five[k] for k in ["Enc_test", "Maint_test", "Maint onto Enc"]}
    for gname, g in [("five groups", five), ("three test groups", three)]:
        H, p, d = kw_dunn(g)
        FIG3D[(rd, gname)] = (H, p, d)
        med = {k: round(float(np.median(v)), 3) for k, v in g.items()}
        print(f"trials ({rd}), {gname}: H = {H:.2f}, p = {p:.3g}; medians {med}")
        print(f"   Dunn BH p: Enc_test vs Maint_test {d.loc['Enc_test', 'Maint_test']:.3g}, Enc_test vs Maint onto Enc "
              f"{d.loc['Enc_test', 'Maint onto Enc']:.3g}, Maint_test vs Maint onto Enc {d.loc['Maint_test', 'Maint onto Enc']:.3g}")
        print(f"   encoding test above maintenance test in {int((g['Enc_test'] > g['Maint_test']).sum())} of 100 splits, "
              f"above maintenance onto encoding in {int((g['Enc_test'] > g['Maint onto Enc']).sum())}; "
              f"maintenance test above maintenance onto encoding in {int((g['Maint_test'] > g['Maint onto Enc']).sum())}")
print("\nranges over splits:", {k: (round(float(v.min()), 2), round(float(v.max()), 2)) for k, v in
                                 {"Enc_train": col("Enc_train"), "Enc_test": col("Enc_test"),
                                  "Maint_test (a)": col(("a", "Maint_test")), "Maint onto Enc (a)": col(("a", "Maint onto Enc"))}.items()})
print("H with five groups of 100 can be at most", round(stats.kruskal(*[np.arange(100) + 100 * i for i in range(5)])[0], 1),
      "; with three groups of 100 at most", round(stats.kruskal(*[np.arange(100) + 100 * i for i in range(3)])[0], 1))
for rd in ["a", "b"]:
    print(f"\ntrials ({rd}), five groups, Dunn BH p:")
    print(FIG3D[(rd, "five groups")][2].map(lambda v: f"{v:.2e}").to_string())
print("paper: H = 457.217 (411.5 to 502.9); Dunn p 2.38e-38 and 1.25e-29; maintenance vs projected p > 0.05")

**Result: reproduced.** With the code's trials (reading b) and its five groups, H = 457.62, within
the tolerance of 411.5 to 502.9. Encoding test separation (median 7.27) is above maintenance test
separation (2.75; Dunn BH p 1.1e-38, the paper's 2.38e-38) and above maintenance projected into
the encoding space (3.02; p 2.39e-29, the paper's 1.25e-29), in all 100 splits. The two
maintenance conditions do not differ (p 0.077). With the same 30 trials in both epochs (reading a)
H = 457.80 and encoding is above both maintenance conditions (p 5.96e-40 and 1.24e-27), but the
two maintenance conditions differ (p 0.019; maintenance in its own space is above the projection
in only 25 of 100 splits). With the three test groups alone H is 207.95 to 213.90, and three
groups of 100 cannot exceed 265.8, so the paper's H comes from the five groups.

## fig3abc-trajectory-complexity

> dPCA encoding trajectories of load 1 concept-cell activity made a single large excursion in state
> space, while maintenance trajectories were more complex, with multiple turns and changes in
> direction.

On the held-out (test) trajectories of the fig3d fits, each stimulus on the first three stimulus
dPCs: (a) tortuosity, path length over the largest distance from the start point; (b) the number of
reversals of direction (sign changes of the time derivative, steps of exactly zero skipped),
summed over the three dPCs. Medians over the 5 stimuli and 100 splits. The maintenance trajectory
is 2.5 times as long as the encoding one (1249 against 499 bins), which by itself lengthens the
path, so the first 499 maintenance bins are shown as a variant.

In [ ]:
def complexity(key, nbins=None):
    tort, rev = [], []
    for r in RES:
        Z = r[key]
        if nbins:
            Z = Z[:, :, :nbins]
        for s in range(Z.shape[1]):
            tort.append(wm.tortuosity(Z[:, s, :]))
            rev.append(wm.reversals(Z[:, s, :]))
    return np.array(tort), np.array(rev)


ENC = complexity("traj_enc")
rows = [{"trajectory": "encoding (499 bins)", "tortuosity_median": np.median(ENC[0]), "reversals_median": np.median(ENC[1]),
         "tortuosity_range": f"{ENC[0].min():.2f}-{ENC[0].max():.2f}", "reversals_range": f"{ENC[1].min()}-{ENC[1].max()}"}]
for rd in ["a", "b"]:
    for nb_, lab in [(None, "1249 bins"), (499, "first 499 bins")]:
        M = complexity((rd, "traj_maint"), nb_)
        u0 = stats.mannwhitneyu(M[0], ENC[0])
        u1 = stats.mannwhitneyu(M[1], ENC[1])
        rows.append({"trajectory": f"maintenance, trials ({rd}), {lab}", "tortuosity_median": np.median(M[0]),
                     "reversals_median": np.median(M[1]), "tortuosity_range": f"{M[0].min():.2f}-{M[0].max():.2f}",
                     "reversals_range": f"{M[1].min()}-{M[1].max()}",
                     "maint_above_enc_tort": bool(np.median(M[0]) > np.median(ENC[0])),
                     "maint_above_enc_rev": bool(np.median(M[1]) > np.median(ENC[1])),
                     "p_tort (rank-sum, 500 vs 500)": u0.pvalue, "p_rev": u1.pvalue})
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("each value is one stimulus of one split: 500 per trajectory type")

**Result: reproduced.** On the held-out trajectories, maintenance has the higher median than
encoding under both measures and both trial readings: tortuosity 4.93 (a) and 5.22 (b) against
2.16, and reversals 23 against 8 (rank-sum p printed as 0). Part of this comes from length: over
the first 499 maintenance bins, the length of the encoding trajectory, the medians are 2.46 and
2.57 against 2.16 (tortuosity) and 9 against 8 (reversals). They are still higher, with rank-sum
p printed as 0 in the table above. The claim is a visual reading and the measures are this
attempt's readings of it.

## fig3h-early-vs-late-maintenance

> Stimulus separation was higher in the first half of maintenance than in the second half, and
> higher than second-half activity projected into the first-half space.

Maintenance split at bin 624; each half its own dPCA fit; late test data also projected into the
early space. Kruskal-Wallis over five groups (the code), then Dunn with BH. Readings: the trials
of fig3d, (a) and (b); the `protect` setting of the early and late fits, the code's (early
`['t']`, late `[]`) and the three other combinations. Paper: H = 417.58 (tolerance 41.76), Dunn
p 9.70e-12 (early against late) and 2.54e-09 (early against late projected into early).

In [ ]:
rows = []
for rd in ["a", "b"]:
    for pe, pl in [("t", "none"), ("none", "none"), ("t", "t"), ("none", "t")]:
        g = {"Early_train": col((rd, pe, "Early_train")), "Late_train": col((rd, pl, "Late_train")),
             "Early_test": col((rd, pe, "Early_test")), "Late_test": col((rd, pl, "Late_test")),
             "Late onto Early": col((rd, pe, "Late onto Early"))}
        H, p, d = kw_dunn(g)
        rows.append({"trials": rd, "protect early/late": f"{pe}/{pl}" + (" (code)" if (pe, pl) == ("t", "none") else ""),
                     "H": round(H, 2), "p": p,
                     "med_early_test": round(float(np.median(g["Early_test"])), 3), "med_late_test": round(float(np.median(g["Late_test"])), 3),
                     "med_late_onto_early": round(float(np.median(g["Late onto Early"])), 3),
                     "dunn_early_vs_late": d.loc["Early_test", "Late_test"], "dunn_early_vs_late_onto_early": d.loc["Early_test", "Late onto Early"],
                     "early_above_late_splits": int((g["Early_test"] > g["Late_test"]).sum()),
                     "zero_fits_early": int((g["Early_train"] < 1e-3).sum()), "zero_fits_late": int((g["Late_train"] < 1e-3).sum())})
H3 = pd.DataFrame(rows)
print(H3.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("zero_fits: splits whose training separation is below 1e-3, a fit whose regularization removed the stimulus signal")
print("paper: H = 417.58 (375.8 to 459.3); Dunn p 9.70e-12 and 2.54e-09")
print(f"notebook ran in {(time.time() - T0) / 60:.1f} min")

**Result: reproduced.** With `protect=[]` in both halves, H = 410 (trials a) and 413 (trials
b), within the tolerance of 375.8 to 459.3. Early test separation (medians 3.99 and 3.53) is
above late separation (3.18 and 2.63; Dunn BH p 4.2e-08 and 7.2e-10) and above late activity
projected into the early space (3.05 and 2.61; p 8.4e-11 and 1.2e-10). With the early half
unprotected and the late half protected, H = 456 and 454, and early is above both late
conditions too (p at most 2.3e-05). The code's own setting (early `['t']`, late `[]`) reverses
the result. There, early test separation (0.97 and 0.58) is below late separation in all 100
splits (H 394 and 380), and 2 or 10 early fits have no stimulus separation at all in training.
So the paper's figure is not what the code as shipped gives.

## Summary

| Claim | Outcome |
|---|---|
| fig3d-encoding-vs-maintenance-separability | reproduced (the code's trials; with the same trials in both epochs the two maintenance conditions differ) |
| fig3abc-trajectory-complexity | reproduced (both measures and both trial readings) |
| fig3h-early-vs-late-maintenance | reproduced with `protect=[]` in the early half; the code's setting reverses it |

How the analysis differs from the paper's. The 100 splits are drawn before the fits and each
split's dPCA is seeded on its own, so the splits are not the code's; the fits run in parallel, one
linear-algebra thread each. The trials, binning, z-scoring, smoothing, dPCA settings, distance
measure and tests follow 08_dpca_matrices.py and the scripts of 03_dpca. The trajectory complexity
measures are the claims file's readings of a visual claim.

What was found. The code's maintenance tensor takes trials by `trial_id` where the encoding tensor
takes them by `new_trial_id` (08_dpca_matrices.py:192 against 105). In 4 of the 14 sessions with
concept cells, these are other trials (9 to 11 of 30 shared, and only 13 of the 30 at load 1).
The paper's Dunn p values are matched most closely under the code's trials. The early-half fit of
main_3h,supp_2c,d,l.py protects the time axis and the late-half fit does not, and with these
settings the early half separates the stimuli less than the late half.